In [22]:
from ipynb.fs.full.utils_ecurves import *
from ipynb.fs.full.utils_ntheory import *

import matplotlib.pyplot as plt
import random

In [24]:
def linearGenerator(E, p, G, U):
    """
    :(E,p: elliptic curve,
    :G: a point on the elliptic curve,
    :U: the seed of the generator
    :return: Q = nG + U
    """
    P = []
    for i in range(p):
        Q = addPoints(doubleAndAdd(G, i, E, p), U, E, p) # P = iG + U
        if Q in P:
            break
        P.append(Q)
    return P

def lastKbits(n, k):
    """
    returns the last k bits of a given number n. 
    If its binary representation doesn't have enough bits,
    it will fill to the left with zeroes.
    """
    nb = bin(n)[2:]
    if len(nb) < k:
        return nb.zfill(k)
    else:
        return nb[-k:]

def pointToLastBits(G, k):
    """
    Takes the k last bits from G's binary representation
    """
    if G == 'O':
        return [0]
    Gx, Gy = lastKbits(G[0], k), lastKbits(G[1], k)
    GxList, GyList = [int(k) for k in Gx], [int(k) for k in Gy]
    return [GxList, GyList]

def pointToBitMod(G):
    """
    Add up G both coordinates and take them modulo 2.
    """
    if G == 'O':
        return 0
    return (G[0] + G[1]) % 2

def pointToBit(Points):
    """
    Generates a bit sequence from a given Point sequence
    """
    return [pointToBitMod(Q) for Q in Points]

In [26]:
def generateRandomSeq(E, p, G, U, n):
    """
    :params (E, p, G): An elliptic curve, a prime number and a point on the curve,
    :param U: the seed for the dynamical system,
    :param n: the amount of bits desired for the sequence

    :return: a nested list of pseudo random bits of length n
    """
    P, B = [0] * n, [0] * n
    P[0], B[0] = U, pointToBitMod(U) 
    for i in range(n - 1):
        P[i+1] = addPoints(G, P[i], E, p)
        B[i+1] = pointToBitMod(P[i])
    return B

def generateKSequences(E, p, G, n, k):
    """
    :params (E, p, G): elliptic curve, a prime number congruent to 3 mod 4, starting point,
    :param n: size of the bit sequence,
    :param k: number of iterations representing the size of the sample,
    :return: a sample of pseudo random bit sequences
    """
    if p % 4 != 3:
        print("Error: the prime p must be congruent to 3 modulo 4")
        return 0
    bits = [[]] * k
    A, B = E
    for i in range(k):
        x = random.randint(1, p)
        a = (x**3 + A*x + B) % p
        while fast2Power(a, (p - 1) // 2, p) != 1:
            x = random.randint(1, p)
            a = (x**3 + A*x + B) % p
        y = fast2Power(a, (p + 1) // 4, p)
        U = [x,y]
        if not pointOnCurve(U, E, p):
            return 0
        bits[i] = generateRandomSeq(E, p, G, U, n)
    return bits




def plotPvals(data, test, m, alpha):
    """
    prints a plot of the p values
    
    :param data: a dictionary containing the pvalues for each test,
    :param test: the name of the test,
    :param  m: sample size
    """
    x = list(range(m))
    y = data[test]
    #p = 1 - alpha
    #e = 3 * sqrt(p * (1 - p) / m)
    #a0, a1 = p - e, p + e
    
    plt.scatter(x, y)
    plt.xlabel('observation')
    plt.ylabel('p-value')
    plt.title("P-value plot for {}".format(test))
    plt.axhline(y=alpha, color='red', linestyle='--', linewidth=2, label='Línea y=5')
    #plt.axhline(y=a1, color='red', linestyle='--', linewidth=2, label='Línea y=5')
    plt.show()

def metrics(data, m, alpha):
    """
    :data: a dictionary test - pvalues,
    :m: sample size,
    :alpha: level of significance,
    :return: table containing proportion of data passed
    """
    tests = ["frequencies", "blocks", "runs"]
    headers = ["Test", "accepted", "rejected", "proportion passed", "confidence"]
    acc, rej, prop, conf = [0]*3, [0]*3, [0]*3, [0]*3
    p = 1 - alpha
    e = 3 * sqrt(p * (1 - p) / m)
    a0, a1 = p - e, p + e
    for i in range(3):
        acc[i] = sum(1 for p in data[tests[i]] if p >= alpha)
        rej[i] = sum(1 for p in data[tests[i]] if p < alpha)
        prop[i] = acc[i]/m
        conf[i] = (prop[i] >= a0) and (prop[i] <= a1)
    d = [[]] * 3
    for j in range(3):
        d[j] = [tests[j], acc[j], rej[j], prop[j], conf[j]]
    print(tabulate(d, headers, tablefmt='grid'))

In [38]:
def randomBitcoins(n,k):
    """
    generates a nested list of k random sequences of bits of size n each.
    :param k: the desired number of Bitcoin random sequences
    :param n: the desired length of each Bitcoin sequence
    """
    E = [0,7]
    p = 2**256 - 2**32 - 977
    x = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
    y = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8
    G = [x, y]
    L = generateKSequences(E, p, G, n, k)
    return L